In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F 

from torch.utils.data import DataLoader
from torchvision.utils import make_grid
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import confusion_matrix
%matplotlib inline

In [2]:
transform = transforms.ToTensor()

In [3]:
train_data = datasets.MNIST(root='../pytorch-for-deep-learning/Data', train=True, download=True, transform=transform)

In [4]:
test_data = datasets.MNIST(root='../pytorch-for-deep-learning/Data', train=False, download=True, transform=transform)

In [5]:
train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: ../pytorch-for-deep-learning/Data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [6]:
train_loader = DataLoader(train_data, batch_size=10, shuffle=True)
test_loader = DataLoader(test_data, batch_size=10, shuffle=False)

In [7]:
# 1 Color Channel, 6 Output Channels, 3x3 kernel, stride 1
conv1 = nn.Conv2d(1, 6, 3, 1) # 6 filters ---> pooling layer ---> conv 2

# 6 Input Channels, 16 Output Channels, 3x3 kernel, stride 1
conv2 = nn.Conv2d(6, 16, 3, 1)

In [8]:
for i, (X_train, y_train) in enumerate(train_data):
    break

In [9]:
x = X_train.view(1,1,28,28) # ---> 4D batch size, color channel, height, width

In [10]:
x = F.relu(conv1(x))

In [11]:
x.shape

torch.Size([1, 6, 26, 26])

In [12]:
x = F.max_pool2d(x, 2, 2) # pooling layer, 2x2 kernel, stride 2

In [13]:
x.shape

torch.Size([1, 6, 13, 13])

In [14]:
x = F.relu(conv2(x))

In [15]:
x.shape

torch.Size([1, 16, 11, 11])

In [16]:
x = F.max_pool2d(x, 2, 2)

In [17]:
x.shape

torch.Size([1, 16, 5, 5])

In [18]:
# 16 output channels * 5 * 5, x.view to flatten the tensor
x.view(-1, 16*5*5).shape

torch.Size([1, 400])

In [19]:
class ConvolutionalNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 3, 1)
        self.conv2 = nn.Conv2d(6, 16, 3, 1)
        self.fc1 = nn.Linear(5*5*16, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
    
    def forward(self, X):
        X = F.relu(self.conv1(X))
        X = F.max_pool2d(X, 2, 2)
        X = F.relu(self.conv2(X))
        X = F.max_pool2d(X, 2, 2)
        X = X.view(-1, 16*5*5)
        X = F.relu(self.fc1(X))
        X = F.relu(self.fc2(X))
        X = self.fc3(X)
        return F.log_softmax(X, dim=1)
    
torch.manual_seed(42)
model = ConvolutionalNetwork()
model

ConvolutionalNetwork(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [20]:
for param in model.parameters():
    print(param.numel())

54
6
864
16
48000
120
10080
84
840
10


In [21]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [22]:
import time
start_time = time.time()

epochs = 5
train_losses = []
test_losses = []
train_correct = []
test_correct = []   

for i in range(epochs):
    trn_corr = 0
    tst_corr = 0
    
    # Run the training batches
    for b, (X_train, y_train) in enumerate(train_loader):
        b+=1
        
        # Apply the model
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
 
        # Tally the number of correct predictions
        predicted = torch.max(y_pred.data, 1)[1]
        batch_corr = (predicted == y_train).sum()
        trn_corr += batch_corr
        
        # Update parameters
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Print interim results
        if b%600 == 0:
            print(f'epoch: {i:2}  batch: {b:4} [{10*b:6}/60000]  loss: {loss.item():10.8f}  \
accuracy: {trn_corr.item()*100/(10*b):7.3f}%')
        
    train_losses.append(loss)
    train_correct.append(trn_corr)
        
    # Run the testing batches
    with torch.no_grad():
        for b, (X_test, y_test) in enumerate(test_loader):

            # Apply the model
            y_val = model(X_test)

            # Tally the number of correct predictions
            predicted = torch.max(y_val.data, 1)[1] 
            tst_corr += (predicted == y_test).sum()
            
    loss = criterion(y_val, y_test)
    test_losses.append(loss)
    test_correct.append(tst_corr)

print(f'\nDuration: {time.time() - start_time:.0f} seconds') # print the time elapsed


epoch:  0  batch:  600 [  6000/60000]  loss: 0.04055632  accuracy:  78.417%
epoch:  0  batch: 1200 [ 12000/60000]  loss: 0.08253475  accuracy:  85.800%
epoch:  0  batch: 1800 [ 18000/60000]  loss: 0.36018974  accuracy:  88.689%
epoch:  0  batch: 2400 [ 24000/60000]  loss: 0.01818694  accuracy:  90.471%
epoch:  0  batch: 3000 [ 30000/60000]  loss: 0.00846514  accuracy:  91.623%
epoch:  0  batch: 3600 [ 36000/60000]  loss: 0.00114300  accuracy:  92.486%
epoch:  0  batch: 4200 [ 42000/60000]  loss: 0.62470806  accuracy:  93.129%
epoch:  0  batch: 4800 [ 48000/60000]  loss: 0.04621773  accuracy:  93.615%
epoch:  0  batch: 5400 [ 54000/60000]  loss: 0.00839211  accuracy:  94.039%
epoch:  0  batch: 6000 [ 60000/60000]  loss: 0.04479276  accuracy:  94.350%
epoch:  1  batch:  600 [  6000/60000]  loss: 0.00351554  accuracy:  97.750%
epoch:  1  batch: 1200 [ 12000/60000]  loss: 0.04203721  accuracy:  97.875%
epoch:  1  batch: 1800 [ 18000/60000]  loss: 0.00102587  accuracy:  97.922%
epoch:  1  b